# Midden-laag spanning Stedin met maximale waterdiepte

In [ ]:
from pathlib import Path
import geopandas as gpd
import folium 
from shapely.geometry import LineString, Polygon, box
import rasterio
import matplotlib.pyplot as plt
from shapely.geometry import shape

import numpy as np
import geopandas as gpd
from scipy.spatial import Voronoi
from shapely.geometry import Polygon

import rasterio
import geopandas as gpd
from shapely.geometry import Point
from rasterio.transform import Affine

In [ ]:
root_dir = Path(r'N:\Projects\11209000\11209175\B. Measurements and calculations\Data\full_run\V_K_analyse')
static_path = root_dir.joinpath("static_maxwd")
network_path = static_path.joinpath("network")
stedin_data = root_dir.joinpath("Stedin_data")
hazard_path = static_path.joinpath("hazard", "processed")
assert root_dir.exists()

In [ ]:
msls_station_path = stedin_data.joinpath("MiddenLaagspanningsstations")
msls_stations = gpd.read_file(msls_station_path.joinpath("MiddenLaagspanningsstations.shp"), driver='ESRI Shapefile')
msls_stations_centroids = msls_stations['geometry'].representative_point()
bounding_box_msls = msls_stations_centroids.unary_union.convex_hull


In [ ]:
#zuidholland = gpd.read_file(zh_path.joinpath("zuidholland.geojson"), driver='GeoJSON')
# mask = zuidholland.to_crs(epsg=28992)
# clipped_points_ls = gpd.clip(ls_stations_centroids, mask)
# clipped_points_ms = gpd.clip(ms_stations_centroids, mask)
# clipped_points_hs = gpd.clip(hs_stations_centroids, mask)

In [ ]:
def voronoi_polygons(gdf, bounding_box):
    points = gdf.geometry.apply(lambda geom: (geom.x, geom.y)).tolist()
    vor = Voronoi(points)
    polygons = []
    for region in vor.regions:
        if not -1 in region and len(region) > 0:
            polygon = Polygon([vor.vertices[i] for i in region])
            clipped_polygon = polygon.intersection(bounding_box)
            polygons.append(clipped_polygon)
    return gpd.GeoDataFrame(geometry=polygons)

# Assuming 'gdf' is your GeoDataFrame with point geometries

gdf_msls = msls_stations_centroids.set_crs("EPSG:28992")
voronoi_gdf_msls = voronoi_polygons(gdf_msls, bounding_box_msls)
voronoi_gdf_msls = voronoi_gdf_msls.set_crs("EPSG:28992")


#voronoi_gdf = voronoi_gdf[voronoi_gdf.is_valid]



# MiddenLaagspanning

TO DO : Implement Daniel's improvements for boundaries 

In [ ]:
# Plotting
m = voronoi_gdf_msls.explore(color='orange', tooltip=False)
gdf_msls.explore(m=m, marker_kwds={'color': 'blue', 'radius': 1})

In [ ]:
import rasterio
import matplotlib.pyplot as plt

hazard_map = hazard_path.joinpath("RMM_merge_max_wd_merge_clipNL_processed.tif")
# Open the TIF file using rasterio
with rasterio.open(hazard_map) as src:
    # Read the TIF file as a numpy array
    tif_array = src.read(1)  # Change the band index (1) if necessary

plt.figure(figsize=(10, 10))
plt.imshow(tif_array, cmap='Blues_r')  # Change the colormap if desired
plt.colorbar(label='Pixel Values')
plt.title('hazard map')
plt.show()  

In [ ]:
raster_map = hazard_path.joinpath("N:\Projects\11209000\11209175\B. Measurements and calculations\Data\full_run\V_K_analyse\static_maxwd\hazard\processed\RMM_merge_max_wd_merge_clipNL_processed.tif")

In [ ]:
# Assuming gdf_msls is already a GeoDataFrame or a DataFrame with geometry data
gdf_msls_2 = gpd.GeoDataFrame()
gdf_msls_2['geometry'] = gdf_msls.geometry
gdf_msls_2

In [ ]:
import os
import rasterio


# Initialize an empty list to store GeoTIFF file names
geotiff_files = []


# Find all GeoTIFF files in the folder
for filename in os.listdir(hazard_path):
    if filename.lower().endswith((".tif", ".tiff")):
        geotiff_files.append(os.path.join(hazard_path, filename))

# Initialize variables for overall extent
min_lon, min_lat = float("inf"), float("inf")
max_lon, max_lat = float("-inf"), float("-inf")

# Loop over the GeoTIFF files
for geotiff_file in geotiff_files:
    with rasterio.open(geotiff_file) as src:
        #print(src.crs)
        bounds = src.bounds
       # print(geotiff_files,src.crs)
        
        min_lon = min(min_lon, bounds.left)
        min_lat = min(min_lat, bounds.bottom)
        max_lon = max(max_lon, bounds.right)
        max_lat = max(max_lat, bounds.top)

# Overall geospatial extent
#print(f"Min Longitude: {min_lon}, Max Longitude: {max_lon}")
#print(f"Min Latitude: {min_lat}, Max Latitude: {max_lat}")

# Create a Shapely polygon
polygon = box(min_lon, min_lat, max_lon, max_lat)

gdf_polygon = gpd.GeoDataFrame(index=[0], crs='epsg:28992', geometry=[polygon])

# Calculate width and height
height = max_lon - min_lon
width = max_lat - min_lat

buffer_polygon_network = polygon.buffer(height)

gdf_buffer_polygon_network = gpd.GeoDataFrame(index=[0], crs='epsg:28992', geometry=[buffer_polygon_network])
gdf_buffer_polygon_network.explore(color='green')

Clippen op de waterkaart

In [ ]:
raster = hazard_map

In [ ]:
import rasterio
import geopandas as gpd

# Read the raster file
with rasterio.open(hazard_map) as src:
    # Read the raster band once
    array = src.read(1)
    raster_crs = src.crs

    # Ensure CRS match between points and raster
    points = gdf_msls_2.to_crs(raster_crs)

    # Clip points to buffer polygon (ensure valid geometry)
    buffer_polygon_network = buffer_polygon_network.buffer(0)
    points_clipped = gpd.clip(points, buffer_polygon_network)

    # Extract coordinates from geometry
    coords = [(geom.x, geom.y) for geom in points_clipped.geometry]

    # Sample raster values at point coordinates
    sampled_values = list(src.sample(coords))

    # Assign sampled values to a new column
    points_clipped['hazard_value'] = [val[0] if val else None for val in sampled_values]


In [ ]:
points_with_hazard = points_clipped[points_clipped['hazard_value'] > 0.2]
points_with_hazard.explore(column='hazard_value', cmap='viridis_r', tiles='CartoDB positron',marker_kwds={'radius': 6})

In [ ]:
import geopandas as gpd

def assign_hazard_to_voronoi(points_with_hazard, voronoi_gdf_msls):
    # Ensure both GeoDataFrames have the same CRS
    if points_with_hazard.crs != voronoi_gdf_msls.crs:
        points_with_hazard = points_with_hazard.to_crs(voronoi_gdf_msls.crs)
    
    # Perform spatial join to find polygons containing points
    joined_gdf = gpd.sjoin(voronoi_gdf_msls, points_with_hazard, how="inner", predicate="contains")
    
    # Group by Voronoi polygon and aggregate hazard values
    voronoi_with_hazard = joined_gdf.groupby(joined_gdf.index).agg({'hazard_value': 'mean'}).reset_index()
    
    # Merge the aggregated hazard values back to the original Voronoi GeoDataFrame
    voronoi_gdf_msls = voronoi_gdf_msls.merge(voronoi_with_hazard, left_index=True, right_on='index', how='left')
    
    return voronoi_gdf_msls

# Example usage
# result = assign_hazard_to_voronoi(points2, voronoi_gdf_msls)


In [ ]:
result = assign_hazard_to_voronoi(points_with_hazard, voronoi_gdf_msls)

In [ ]:

result_with_hazard = result[result['hazard_value']>0.2]
result_with_hazard.explore(column='hazard_value',cmap='viridis_r', tiles='CartoDB positron')

In [ ]:
result.to_file(network_path.joinpath("msls_voronoi_hazard.gpkg"), driver='GeoJSON')